# Kiểm tra cấu trúc UR Fall (bản mới: 1 thư mục cha, các thư mục chuỗi nằm ngay bên trong)

Cấu trúc cần kiểm tra:
```
UR_fall_detection_dataset/
├── adl-01-cam0-rgb/   (ảnh frame của 1 chuỗi)
├── ...
├── adl-40-cam0-rgb/
├── fall-01-cam0-rgb/
└── fall-30-cam0-rgb/
```
**Notebook kiểm tra:** (1) tự tìm thư mục cha; (2) đủ 30 fall + 40 ADL, không trùng/thiếu số thứ tự; (3) ảnh nằm trực tiếp trong thư mục chuỗi hay còn lồng thêm 1 tầng; (4) tên ảnh, đánh số frame, tính liên tục, kích thước, định dạng; (5) **đối chiếu với 2 file CSV nhãn** (số ảnh ↔ số dòng, mẫu nhãn, khoảng ngã); (6) ảnh ghép để xem bằng mắt.

Notebook chỉ **đọc** dữ liệu. **Cách dùng:** Add Input → Run All → đọc phần *Tổng kết*.

In [ ]:
import os, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# ===== CẤU HÌNH (để None thì tự tìm trong /kaggle/input) =====
DATA_ROOT = os.environ.get("URFD_DATA_ROOT")      # để None = tự tìm; muốn điền tay thì gán chuỗi trực tiếp: DATA_ROOT = "/kaggle/input/.../UR_..." (KHÔNG bọc trong os.environ.get)
CSV_FALLS = os.environ.get("URFD_CSV_FALLS")      # điền tay: CSV_FALLS = "/kaggle/input/.../urfall-cam0-falls.csv"
CSV_ADLS  = os.environ.get("URFD_CSV_ADLS")       # điền tay: CSV_ADLS = "/kaggle/input/.../urfall-cam0-adls.csv"
FPS = 30
EXPECT = dict(fall=30, adl=40, size=(640, 480))
MARGIN = 15; N_ADL_SHOW = 12
OUT_DIR = Path("/kaggle/working/check_urfd2") if Path("/kaggle").exists() else Path("./check_urfd2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Tìm thư mục cha

In [ ]:
DIR_RE = re.compile(r"^(adl|fall)[-_]?(\d+)[-_]?(cam\d)?", re.I)

def is_seq_dir(p): return p.is_dir() and DIR_RE.match(p.name) is not None

def find_root():
    if DATA_ROOT: return Path(DATA_ROOT)
    base = Path("/kaggle/input")
    best, best_n = None, 0
    if base.exists():
        for dp, dn, _ in os.walk(base):
            n = sum(1 for d in dn if DIR_RE.match(d))
            if n > best_n: best, best_n = Path(dp), n
            if len(Path(dp).relative_to(base).parts) > 4: dn[:] = []
    if best is None: raise FileNotFoundError("Không thấy thư mục chứa các thư mục adl-XX/fall-XX. Điền DATA_ROOT.")
    return best

ROOT = find_root()
kids = sorted(ROOT.iterdir())
print("Thư mục cha:", ROOT)
print("Số mục con :", len(kids), "| thư mục chuỗi:", sum(is_seq_dir(k) for k in kids))
print("Mục con KHÔNG phải thư mục chuỗi:", [k.name for k in kids if not is_seq_dir(k)][:15] or "(không có)")
print("5 mục đầu:", [k.name for k in kids[:5]], "| 5 mục cuối:", [k.name for k in kids[-5:]])

## 2. Quét từng chuỗi
- `depth` = số tầng thư mục giữa thư mục chuỗi và file ảnh (0 = ảnh nằm ngay trong thư mục chuỗi; 1 = còn 1 thư mục con nữa, như bản cũ).
- Frame number lấy từ cuối tên file (`adl-01-cam0-rgb-001.png` → 1).

In [ ]:
IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp"}
def frame_no(stem):
    nums = re.findall(r"\d+", stem); return int(nums[-1]) if nums else None

rows, frames_map = [], {}
for clip in filter(is_seq_dir, kids):
    m = DIR_RE.match(clip.name)
    cls, num = m.group(1).lower(), int(m.group(2))
    cam = (m.group(3) or "cam0").lower()
    seq = f"{cls}-{num:02d}"
    files = [p for p in clip.rglob("*") if p.is_file()]
    imgs = [p for p in files if p.suffix.lower() in IMG_EXT]
    others = sorted({p.suffix.lower() for p in files} - IMG_EXT)
    depths = {len(p.relative_to(clip).parts) - 1 for p in imgs}
    pairs = sorted((frame_no(p.stem), p) for p in imgs if frame_no(p.stem) is not None)
    fn = [f for f, _ in pairs]
    w = h = None
    if pairs:
        with Image.open(pairs[0][1]) as im: w, h = im.size
    name_ok = all(p.stem.lower().startswith(f"{cls}-{num:02d}") for _, p in pairs[:50]) if pairs else False
    rows.append(dict(seq=seq, cam=cam, cls=cls, folder=clip.name, n_img=len(pairs), n_noimg_files=len(files) - len(imgs),
                     other_ext=",".join(others), depth=max(depths) if depths else None, mixed_depth=len(depths) > 1,
                     f_min=min(fn) if fn else None, f_max=max(fn) if fn else None,
                     contiguous=bool(fn) and len(set(fn)) == max(fn) - min(fn) + 1, dup=len(fn) != len(set(fn)),
                     w=w, h=h, ext=pairs[0][1].suffix.lower() if pairs else None, name_ok=name_ok,
                     example=pairs[0][1].name if pairs else None))
    frames_map[(seq, cam)] = pairs

inv = pd.DataFrame(rows).sort_values(["cls", "seq"]).reset_index(drop=True)
print(inv.groupby(["cls", "cam"]).size().rename("so_chuoi").to_string())
print("\nĐộ sâu ảnh (0 = ảnh nằm ngay trong thư mục chuỗi):", inv.depth.value_counts().to_dict())
print("Kích thước ảnh:", inv.groupby(["w", "h"]).size().to_dict(), "| Định dạng:", inv.ext.value_counts().to_dict())
print("Số ảnh/chuỗi: min={}, median={}, max={}, tổng={}".format(inv.n_img.min(), int(inv.n_img.median()), inv.n_img.max(), inv.n_img.sum()))
print("Frame bắt đầu từ:", inv.f_min.value_counts().to_dict())
print("File không phải ảnh:", inv[inv.n_noimg_files > 0][["seq", "other_ext", "n_noimg_files"]].to_dict("records") or "(không có)")
print("Ví dụ tên file:", inv.example.iloc[0], "...", inv.example.iloc[-1])

# thiếu / trùng số thứ tự
for cls, n in (("fall", EXPECT["fall"]), ("adl", EXPECT["adl"])):
    got = set(int(s.split("-")[1]) for s in inv[(inv.cls == cls) & (inv.cam == "cam0")].seq)
    print(f"{cls}: có {len(got)}/{n} chuỗi; thiếu số: {sorted(set(range(1, n + 1)) - got) or 'không'}; thừa số: {sorted(got - set(range(1, n + 1))) or 'không'}")
dups = inv[inv.duplicated(["seq", "cam"], keep=False)]
print("Chuỗi trùng:", "không có" if dups.empty else dups.to_string())

bad = inv[(inv.n_img == 0) | (~inv.contiguous) | inv.dup | inv.mixed_depth | (~inv.name_ok) | (inv.cam != "cam0")]
print("\n⚠ Chuỗi bất thường (rỗng / frame không liên tục / trùng số / lẫn tầng thư mục / tên ảnh không khớp tên chuỗi / không phải cam0):")
print(bad[["seq", "n_img", "f_min", "f_max", "contiguous", "dup", "mixed_depth", "name_ok", "cam"]].to_string() if len(bad) else "  (không có)")
inv.head(8)

## 3. Biểu đồ số ảnh mỗi chuỗi

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3.2))
d = inv[inv.cam == "cam0"].sort_values(["cls", "seq"])
ax.bar(range(len(d)), d.n_img, color=np.where(d.cls == "fall", "tab:red", "tab:blue"))
ax.set_xticks(range(len(d))); ax.set_xticklabels(d.seq, rotation=90, fontsize=6)
ax.set_ylabel("số ảnh"); ax.set_title("Số frame mỗi chuỗi (đỏ = fall, xanh = ADL)")
plt.tight_layout(); fig.savefig(OUT_DIR / "frames_per_seq.png", dpi=110); plt.show()

## 4. Đọc 2 file CSV nhãn (bắt buộc)
File gốc **không có dòng tiêu đề**. Cột: tên chuỗi, số frame, nhãn (`-1` chưa nằm, `0` đang ngã, `1` đã nằm), rồi các đặc trưng độ sâu (HeightWidthRatio, ...). Cần Add Input chứa `urfall-cam0-falls.csv` và `urfall-cam0-adls.csv`.

In [ ]:
def seq_key(name):
    m = DIR_RE.search(str(name))
    return f"{m.group(1).lower()}-{int(m.group(2)):02d}" if m else None

def find_csv(kind, explicit):
    if explicit: return Path(explicit)
    base = Path("/kaggle/input"); cands = []
    for root in ([base] if base.exists() else []) + [Path(".")]:
        cands += [p for p in root.rglob("*.csv") if p.name.lower().endswith(kind + ".csv") and "cam0" in p.name.lower()]
    return cands[0] if cands else None

# khớp theo đuôi tên file (...-falls.csv / ...-adls.csv): "urfall" cũng chứa chữ "fall" nên không được khớp theo chuỗi con
F_FALLS, F_ADLS = find_csv("falls", CSV_FALLS), find_csv("adls", CSV_ADLS)
if F_FALLS is None or F_ADLS is None:
    allcsv = [str(p) for p in Path("/kaggle/input").rglob("*.csv")] if Path("/kaggle/input").exists() else []
    raise FileNotFoundError("Không thấy file CSV nhãn (falls: %s, adls: %s). Add Input dataset chứa urfall-cam0-falls.csv và urfall-cam0-adls.csv, "
                            "hoặc điền CSV_FALLS / CSV_ADLS. CSV hiện có trong /kaggle/input: %s" % (F_FALLS, F_ADLS, allcsv[:20]))

BASE_COLS = ["seq", "frame", "label", "HeightWidthRatio", "MajorMinorRatio", "BoundingBoxOccupancy", "MaxStdXZ", "HHmaxRatio", "H", "D", "P40"]
def read_label_csv(path):
    d = pd.read_csv(path, header=None)
    d.columns = BASE_COLS[:d.shape[1]] + [f"x{i}" for i in range(max(0, d.shape[1] - len(BASE_COLS)))]
    d["frame"] = pd.to_numeric(d["frame"], errors="coerce"); d = d.dropna(subset=["frame"]).copy()
    d["frame"] = d.frame.astype(int); d["label"] = pd.to_numeric(d["label"], errors="coerce").astype(int)
    d["seq"] = d["seq"].astype(str).map(seq_key)
    if "HeightWidthRatio" in d: d["HeightWidthRatio"] = pd.to_numeric(d["HeightWidthRatio"], errors="coerce")
    return d.sort_values(["seq", "frame"]).reset_index(drop=True)

assert Path(F_FALLS).resolve() != Path(F_ADLS).resolve(), f"CSV_FALLS và CSV_ADLS trỏ cùng 1 file: {F_FALLS}"
csv_f, csv_a = read_label_csv(F_FALLS), read_label_csv(F_ADLS)
assert csv_f.seq.str.startswith("fall").all(), f"falls.csv chứa chuỗi không phải fall (sai file?): {sorted(set(csv_f.seq[~csv_f.seq.str.startswith('fall')]))[:5]}"
assert csv_a.seq.str.startswith("adl").all(), f"adls.csv chứa chuỗi không phải adl (sai file?): {sorted(set(csv_a.seq[~csv_a.seq.str.startswith('adl')]))[:5]}"
csv_all = pd.concat([csv_f, csv_a], ignore_index=True)
print("falls.csv:", F_FALLS, f"-> {len(csv_f)} dòng, {csv_f.seq.nunique()} chuỗi")
print("adls.csv :", F_ADLS, f"-> {len(csv_a)} dòng, {csv_a.seq.nunique()} chuỗi")
print("Nhãn falls.csv:", csv_f.label.value_counts().to_dict(), "| nhãn adls.csv:", csv_a.label.value_counts().to_dict())
print("Tiền tố lạ trong falls.csv:", sorted(s for s in csv_f.seq.dropna().unique() if not s.startswith("fall")) or "không",
      "| trong adls.csv:", sorted(s for s in csv_a.seq.dropna().unique() if not s.startswith("adl")) or "không")
print("Frame nhỏ nhất trong CSV:", csv_all.groupby("seq").frame.min().value_counts().to_dict())

## 5. Đối chiếu số ảnh ↔ số dòng CSV (cam0)
Bước quan trọng nhất. `miss_head/miss_tail/gap_mid` = số frame ảnh **không có nhãn** ở đầu/cuối/giữa.
Trạng thái: `OK` khớp hoàn toàn · `KHOP_SO_LUONG_LECH_DANH_SO` · `CSV_THIEU_FRAME` (CSV là tập con của ảnh; ở bản cũ gặp 18 ADL, chấp nhận cho ADL) · `LECH_THAT` (CSV nhắc frame không có ảnh, nghiêm trọng) · `THIEU_ANH` / `THIEU_CSV`.

In [ ]:
img0 = inv[inv.cam == "cam0"].drop_duplicates("seq").set_index("seq")
csv_sum = csv_all.groupby("seq").agg(csv_rows=("frame", "size"), csv_fmin=("frame", "min"), csv_fmax=("frame", "max"),
                                     csv_dup=("frame", lambda s: s.duplicated().any()))
cmp = img0[["n_img", "f_min", "f_max", "contiguous"]].join(csv_sum, how="outer")
cmp["cls_dir"] = ["Fall" if str(s).startswith("fall") else "ADL" for s in cmp.index]

def csv_not_in_img(seq):
    pr = frames_map.get((seq, "cam0"))
    return np.nan if not pr else len(set(csv_all.loc[csv_all.seq == seq, "frame"]) - {f for f, _ in pr})
cmp["csv_not_in_img"] = [csv_not_in_img(s) for s in cmp.index]
cmp["miss_head"] = cmp.csv_fmin - cmp.f_min
cmp["miss_tail"] = cmp.f_max - cmp.csv_fmax
cmp["gap_mid"] = cmp.csv_fmax - cmp.csv_fmin + 1 - cmp.csv_rows

def status(r):
    if pd.isna(r.n_img): return "THIEU_ANH"
    if pd.isna(r.csv_rows): return "THIEU_CSV"
    if r.csv_not_in_img > 0: return "LECH_THAT"
    if r.n_img == r.csv_rows and r.f_min == r.csv_fmin and r.f_max == r.csv_fmax: return "OK"
    if r.n_img == r.csv_rows: return "KHOP_SO_LUONG_LECH_DANH_SO"
    return "CSV_THIEU_FRAME"
cmp["status"] = cmp.apply(status, axis=1)
cmp["diff"] = cmp.n_img - cmp.csv_rows
print(cmp.status.value_counts().to_string())
print("\nTheo lớp:\n", pd.crosstab(cmp.cls_dir, cmp.status).to_string())
print("\nChuỗi không OK:")
cmp.loc[cmp.status != "OK", ["cls_dir", "n_img", "csv_rows", "f_min", "f_max", "csv_fmin", "csv_fmax", "miss_head", "miss_tail", "gap_mid", "csv_not_in_img", "status"]]

## 6. Phân tích nhãn → `fall_start`, `fall_end`
`fall_start` = frame đầu có nhãn `0` (không có nhãn 0 thì frame đầu có nhãn `1`); `fall_end` = frame đầu có nhãn `1`. Chỉ chuỗi **Fall** có sự kiện ngã; ADL có nhãn 0/1 là cúi/bò/nằm giường/sofa → `adl_hard_neg` (negative khó, không phải lỗi).

In [ ]:
def rle(vals):
    out = []
    for v in vals:
        v = int(v)
        if not out or out[-1] != v: out.append(v)
    return out

recs = []
for seq, g in csv_all.groupby("seq"):
    g = g.sort_values("frame").reset_index(drop=True); lab = g.label.to_numpy()
    i0 = int(np.argmax(lab == 0)) if (lab == 0).any() else None
    i1 = int(np.argmax(lab == 1)) if (lab == 1).any() else None
    s_row = i0 if i0 is not None else i1
    hwr = g.HeightWidthRatio.where(g.HeightWidthRatio > 0)
    recs.append(dict(seq=seq, cls="Fall" if seq.startswith("fall") else "ADL", n_rows=len(g),
        n_neg1=int((lab == -1).sum()), n_0=int((lab == 0).sum()), n_1=int((lab == 1).sum()), pattern=str(rle(lab)),
        start_row=s_row, end_row=i1,
        fall_start_frame=int(g.frame[s_row]) if s_row is not None else None,
        fall_end_frame=int(g.frame[i1]) if i1 is not None else None,
        hwr_standing=hwr[g.label == -1].median() if (lab == -1).any() else np.nan,
        hwr_lying=hwr[g.label == 1].median() if (lab == 1).any() else np.nan))
lab_df = pd.DataFrame(recs).set_index("seq")
lab_df["fall_len_s"] = ((lab_df.fall_end_frame - lab_df.fall_start_frame) / FPS).round(2)
lab_df["hwr_drop"] = (lab_df.hwr_lying / lab_df.hwr_standing).round(2)
isF, isA = lab_df.cls == "Fall", lab_df.cls == "ADL"
lab_df["flag"] = ""
lab_df.loc[isF & (lab_df.n_0 + lab_df.n_1 == 0), "flag"] += "[FALL_KHONG_CO_NHAN_0/1] "
lab_df.loc[isF & (lab_df.n_1 == 0), "flag"] += "[FALL_KHONG_THAY_NAM] "
lab_df.loc[isF & ~lab_df.pattern.str.startswith("[-1"), "flag"] += "[FALL_KHONG_BAT_DAU_O_TU_THE_KHONG_NAM] "
lab_df.loc[isF & (lab_df.hwr_drop >= 0.8), "flag"] += "[HWR_KHONG_GIAM_NHIEU] "
lab_df["adl_hard_neg"] = isA & ((lab_df.n_0 > 0) | (lab_df.n_1 > 0))

print("Mẫu nhãn FALL:\n" + lab_df[isF].pattern.value_counts().to_string())
print("\nMẫu nhãn ADL:\n" + lab_df[isA].pattern.value_counts().to_string())
print(f"\nFall: {isF.sum()} (mong đợi 30) | ADL: {isA.sum()} (mong đợi 40) | ADL hard-negative: {int(lab_df.adl_hard_neg.sum())}")
print("Độ dài nhãn 0 (số frame) ở Fall:", lab_df[isF].n_0.value_counts().to_dict())
print("Thời lượng ngã (s):\n" + lab_df[isF].fall_len_s.describe().round(2).to_string())
lab_df[isF][["n_rows", "n_neg1", "n_0", "n_1", "pattern", "fall_start_frame", "fall_end_frame", "fall_len_s", "hwr_drop", "flag"]]

## 7. Xem bằng mắt
**Fall** (mỗi hàng 5 ảnh): trước ngã → bắt đầu ngã → giữa → vừa nằm → sau đó. Cần thấy đứng → đổ → nằm sàn. Đồ thị HeightWidthRatio phải giảm mạnh đúng chỗ nhãn 0→1 (cam = đang ngã, đỏ = đã nằm).

In [ ]:
def path_for_row(seq, i, g):
    pairs = frames_map.get((seq, "cam0"))
    if not pairs: return None
    if len(pairs) == len(g): return pairs[i][1]
    return dict(pairs).get(int(g.frame.iloc[i]))

def montage_rows(seq):
    g = csv_all[csv_all.seq == seq].sort_values("frame").reset_index(drop=True); r = lab_df.loc[seq]; n = len(g)
    if pd.isna(r.start_row): return g, [0, n // 4, n // 2, 3 * n // 4, n - 1]
    s = int(r.start_row); e = int(r.end_row) if not pd.isna(r.end_row) else s
    return g, [max(s - MARGIN, 0), s, (s + e) // 2, e, min(e + MARGIN, n - 1)]

def show_montage(seqs, fname):
    fig, axes = plt.subplots(len(seqs), 5, figsize=(15, 3.1 * len(seqs)), squeeze=False)
    for r_i, seq in enumerate(seqs):
        g, idxs = montage_rows(seq)
        for c_i, i in enumerate(idxs):
            ax = axes[r_i][c_i]; ax.axis("off"); p = path_for_row(seq, i, g)
            if p is not None: ax.imshow(Image.open(p))
            ax.set_title(f"{seq} | f={int(g.frame.iloc[i])} | nhãn={int(g.label.iloc[i])}", fontsize=9)
    plt.tight_layout(); fig.savefig(OUT_DIR / fname, dpi=100); plt.show()

fall_seqs = sorted(lab_df[lab_df.cls == "Fall"].index)
for k in range(0, len(fall_seqs), 5):
    show_montage(fall_seqs[k:k + 5], f"fall_montage_{k // 5 + 1:02d}.png")

def plot_hwr(seqs, ncols=5, fname="hwr_falls.png"):
    if not seqs: print("Không có chuỗi Fall để vẽ."); return
    nrows = int(np.ceil(len(seqs) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 2.4 * nrows), squeeze=False)
    for ax in axes.ravel(): ax.axis("off")
    for ax, seq in zip(axes.ravel(), seqs):
        ax.axis("on"); g = csv_all[csv_all.seq == seq].sort_values("frame")
        ax.plot(g.frame, g.HeightWidthRatio.where(g.HeightWidthRatio > 0), lw=1)
        for lab, col in ((0, "orange"), (1, "red")):
            m = (g.label == lab).to_numpy()
            if m.any(): ax.fill_between(g.frame, 0, 1, where=m, color=col, alpha=0.25, transform=ax.get_xaxis_transform())
        ax.set_title(seq, fontsize=9); ax.tick_params(labelsize=7)
    plt.tight_layout(); fig.savefig(OUT_DIR / fname, dpi=100); plt.show()
plot_hwr(fall_seqs)

## 8. ADL có nhãn 0/1 (negative khó)
Mỗi hàng: khung đầu → khung bắt đầu đổi nhãn → +10 frame → khung cuối. Xác nhận đây là cúi/bò/nằm giường/sofa, **không phải** té ngã.

In [ ]:
adl_check = lab_df[lab_df.adl_hard_neg].sort_values("n_0", ascending=False)
adl_pick = list(adl_check.index[:N_ADL_SHOW])
print(f"{len(adl_check)} ADL có nhãn 0/1; xem {len(adl_pick)}:", adl_pick)
if adl_pick:
    fig, axes = plt.subplots(len(adl_pick), 4, figsize=(12, 2.9 * len(adl_pick)), squeeze=False)
    for r_i, seq in enumerate(adl_pick):
        g = csv_all[csv_all.seq == seq].sort_values("frame").reset_index(drop=True); s = int(lab_df.loc[seq].start_row)
        for c_i, i in enumerate([0, s, min(s + 10, len(g) - 1), len(g) - 1]):
            ax = axes[r_i][c_i]; ax.axis("off"); p = path_for_row(seq, i, g)
            if p is not None: ax.imshow(Image.open(p))
            ax.set_title(f"{seq} | f={int(g.frame.iloc[i])} | nhãn={int(g.label.iloc[i])}", fontsize=9)
    plt.tight_layout(); fig.savefig(OUT_DIR / "adl_hardneg_montage.png", dpi=100); plt.show()

## 9. Tổng kết và xuất file
Điền `MANUAL_NOT_FALL` (chuỗi Fall bạn xem ảnh thấy **không** có cảnh ngã) nếu có. Xuất: `urfd_inventory.csv`, `urfd_check_report.csv` (bảng đối chiếu + cờ), `urfd_fall_intervals.csv` (khoảng ngã của Fall đạt kiểm tra → đầu vào `metadata.csv`).

In [ ]:
MANUAL_NOT_FALL = []      # vd ["fall-07"]
MANUAL_NOTES = {}

inv.to_csv(OUT_DIR / "urfd_inventory.csv", index=False)
report = cmp.join(lab_df.drop(columns=["cls"]), how="outer")
report["manual_not_fall"] = report.index.isin(MANUAL_NOT_FALL)
report["manual_note"] = report.index.map(lambda s: MANUAL_NOTES.get(s, ""))
struct_ok = inv.set_index("seq").loc[lambda d: d.cam == "cam0"]
struct_ok = (struct_ok.n_img > 0) & struct_ok.contiguous & ~struct_ok.dup
report["struct_ok"] = struct_ok.reindex(report.index).fillna(False)
status_ok = np.where(report.cls_dir == "Fall", report.status == "OK", report.status.isin(["OK", "CSV_THIEU_FRAME"]))
report["ok_for_use"] = pd.Series(status_ok, index=report.index) & (report.flag.fillna("") == "") & report.struct_ok & ~report.manual_not_fall
report.to_csv(OUT_DIR / "urfd_check_report.csv")
fall_ok = report[(report.cls_dir == "Fall") & report.ok_for_use][["fall_start_frame", "fall_end_frame", "n_img"]].copy()
fall_ok.insert(0, "fps", FPS); fall_ok.to_csv(OUT_DIR / "urfd_fall_intervals.csv")

c0 = inv[inv.cam == "cam0"]
print("========== TỔNG KẾT ==========")
print(f"Thư mục cha : {ROOT}")
print(f"Cấu trúc    : fall={int((c0.cls=='fall').sum())}/30, adl={int((c0.cls=='adl').sum())}/40, tổng ảnh={int(inv.n_img.sum())}, "
      f"depth={inv.depth.value_counts().to_dict()}, kích thước={inv.groupby(['w','h']).size().to_dict()}")
print(f"Ảnh↔CSV     : {report.status.value_counts().to_dict()}")
print(f"Fall dùng được: {len(fall_ok)}/{int((report.cls_dir=='Fall').sum())} | ADL dùng được: {int(((report.cls_dir=='ADL') & report.ok_for_use).sum())}/{int((report.cls_dir=='ADL').sum())} | ADL hard-neg: {int(lab_df.adl_hard_neg.sum())}")
probs = report[~report.ok_for_use][["cls_dir", "status", "n_img", "csv_rows", "pattern", "flag"]]
print("\nChuỗi cần xử lý/xem lại:\n" + probs.to_string() if len(probs) else "\nKhông có chuỗi nào bị cờ.")
print("\nĐã ghi:", sorted(p.name for p in OUT_DIR.iterdir()))